# UL2 on SVAMP (300 samples) - Colab Runner

This notebook runs the repo pipeline with `google/ul2` on `svamp` using `max_samples=300`.

Caching is configured so generation outputs persist while Hugging Face cache remains ephemeral:
- Pipeline generation cache JSONL is persisted on Google Drive (reuses already-generated samples across reruns).
- Hugging Face model/dataset cache uses Colab local disk (fresh each new session).

In [ ]:
!pip -q install datasets openai tqdm pandas matplotlib transformers torch

In [ ]:
# Clone or update your repo branch.
import os

REPO_URL = 'https://github.com/ieliashberg/ECS189G_Self_Consistency'
REPO_DIR = 'ECS189G_Self_Consistency'
BRANCH = 'ilan/testing'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
    %cd /content

%cd {REPO_DIR}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Colab + Drive cache configuration.
import os
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/ECS189G_Self_Consistency'
DRIVE_RESULTS_DIR = f'{DRIVE_ROOT}/results'
DRIVE_CACHE_DIR = f'{DRIVE_RESULTS_DIR}/cache'

# Hugging Face cache stays on Colab local disk (ephemeral across sessions).

# Optional: set HF token from Colab secrets if available.
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

CACHE_PATH = f'{DRIVE_CACHE_DIR}/generation_cache_ul2_svamp_300.jsonl'
OUTPUT_CSV = f'{DRIVE_RESULTS_DIR}/self_consistency_results/ul2_svamp_300_with_greedy.csv'

# If False, cache file is deleted and generation restarts from scratch.
RESUME_FROM_GENERATION_CACHE = True

for path in [DRIVE_RESULTS_DIR, DRIVE_CACHE_DIR, f'{DRIVE_RESULTS_DIR}/self_consistency_results']:
    Path(path).mkdir(parents=True, exist_ok=True)

if not RESUME_FROM_GENERATION_CACHE and Path(CACHE_PATH).exists():
    Path(CACHE_PATH).unlink()

print('CACHE_PATH:', CACHE_PATH)
print('OUTPUT_CSV:', OUTPUT_CSV)

In [ ]:
# Run UL2 on SVAMP with exactly 300 examples.
from run_pipeline import build_pipeline_config, run_benchmark_pipeline
from pipeline_io import print_results_summary

MODEL = 'google/ul2'
DATASETS = ['svamp']
METHODS = ['greedy', 'self_consistency']
K_VALUES = [40]
MAX_SAMPLES = 300
SELF_CONSISTENCY_TEMPERATURE = 0.7
# In this pipeline, greedy always runs with temperature=0.0 and k=1.
# SELF_CONSISTENCY_TEMPERATURE is used only for self_consistency.

config = build_pipeline_config(
    model=MODEL,
    datasets=DATASETS,
    methods=METHODS,
    k_values=K_VALUES,
    max_samples=MAX_SAMPLES,
    self_consistency_temperature=SELF_CONSISTENCY_TEMPERATURE,
    cache_path=CACHE_PATH,
    output_csv=OUTPUT_CSV,
)

rows = run_benchmark_pipeline(config)
print_results_summary(rows, config.output_csv)

In [ ]:
# Optional: check how many cache rows currently exist for this exact run key.
import json
from pathlib import Path

cache_file = Path(CACHE_PATH)
counts = {'greedy': 0, 'self_consistency': 0}
if cache_file.exists():
    with cache_file.open('r', encoding='utf-8') as f:
        for line in f:
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if row.get('model') == MODEL and row.get('dataset') == 'svamp':
                method = row.get('method')
                if method in counts:
                    counts[method] += 1

for method_name, count in counts.items():
    print(f'Cached rows for model={MODEL}, dataset=svamp, method={method_name}: {count}')